In [ ]:
import pandas as pd
import numpy as np

## 1. Business case & Data Collection

## 2.Data Understanding

### 2.1 Exploratorio Inicial

In [ ]:
#Cargo el dataframe desde un archivo CSV
df = pd.read_csv("./data/steel-faults.csv", sep=",")

In [ ]:
#Obtengo el numero de filas y columnas de la tabla (utilizo una sola tabla)
print("Número de filas y columnas:", df.shape)

In [ ]:
#Saco la información del dataframe (tipo de datos, cantidad de valores no nulos, etc.)
df.info()

In [ ]:
#Muestro los primeros registros del dataframe
df.head()

In [ ]:
#Muestro los últimos registros del dataframe
df.tail()

In [ ]:
#Cambio a español el nombre de las columnas que luego mostraré en los gráficos
df = df.rename(columns={
    'TypeOfSteel_A300': 'A300', 
    'TypeOfSteel_A400': 'A400', 
    'Steel_Plate_Thickness': 'Espesor',
    'Pastry': 'Pliegue',
    'Z_Scratch': 'Raya', 
    'K_Scatch': 'Surco',
    'Stains': 'Mancha',
    'Dirtiness': 'Suciedad',
    'Bumps': 'Abolladura', 
    'Other_Faults': 'Otro'
    'Pixels_Areas': 'Area_Pixel', 
    'X_Perimeter': 'Perimetro_X', 
    'Sum_of_Luminosity': 'Sum_Luminosidad', 
    'Luminosity_Index': 'Indice_Luminosidad', 
    'Square_Index': 'Indice_Forma_Cuadrada'
})

### 2.2 Tabla de Variables

In [ ]:
#Defino las listas de vbles en fn de su naturaleza
coordenadas = ['X_Minimum', 'X_Maximum', 'Y_Minimum', 'Y_Maximum']
magnitud = ['Pixels_Areas', 'X_Perimeter', 'Y_Perimeter', 'LogOfAreas', 'Log_X_Index', 'Log_Y_Index']
intensidad = ['Sum_of_Luminosity', 'Minimum_of_Luminosity', 'Maximum_of_Luminosity', 'Luminosity_Index']
maq_mat = ['Length_of_Conveyer', 'A300', 'A400', 'Espesor']
forma = ['Edges_Index', 'Empty_Index', 'Square_Index', 'Outside_X_Index', 'Edges_X_Index', 'Edges_Y_Index', 'Outside_Global_Index', 'Orientation_Index', 'SigmoidOfAreas']
defecto = ['Pliegue', 'Raya', 'Surco', 'Mancha', 'Suciedad', 'Abolladura', 'Otro']

In [ ]:
#importo las funciones del archivo funciones.py
import sys
import os
sys.path.append(os.path.abspath('../'))
from utils.funciones import asignar_naturaleza, asignar_definicion

In [ ]:
#Construyo el inventario
tabla_vbles = pd.DataFrame({
    'Variable': df.columns,
    'Tipo de Dato': df.dtypes.values,
    'Naturaleza': [asignar_naturaleza(c, coordenadas, magnitud, intensidad, maq_mat, forma, defecto) for c in df.columns],
    'Descripción': [asignar_definicion(c, coordenadas, magnitud, intensidad, maq_mat, forma, defecto) for c in df.columns]
})

tabla_vbles.sort_values(by='Naturaleza').reset_index(drop=True)

## 3.Data Cleaning

In [ ]:
#Como ya vi en la info() del df que no hay nulos, compruebo ahora si tengo duplicados por si acaso
#importo la funcion
from utils.funciones import duplicados
duplicados(df)

In [ ]:
#Compruebo si tengo outliers
from utils.funciones import detectar_outliers
columnas_a_analizar = coordenadas + magnitud + intensidad + maq_mat + forma
tabla_outliers = detectar_outliers(df, columnas_a_analizar)
print(tabla_outliers)

In [ ]:
#Creo una columna categórica para el tipo de acero
from utils.funciones import asignar_tipo_acero
df['Tipo_Acero'] = df.apply(asignar_tipo_acero, axis=1)
print(df['Tipo_Acero'].value_counts())
df[['A300', 'A400', 'Tipo_Acero']].head()

In [ ]:
#creo una columna categorica para el tipo de defecto
from utils.funciones import asignar_tipo_defecto
df['Tipo_Defecto'] = df.apply(lambda fila: asignar_tipo_defecto(fila, defecto), axis=1)
print(df['Tipo_Defecto'].value_counts())
df[defecto + ['Tipo_Defecto']].head()

In [ ]:
#Creo una tabla Tipo de Acero vs Tipo de Defecto
tabla_1 = pd.crosstab(df['Tipo_Acero'], df['Tipo_Defecto'], margins=True)

#Ordeno los nombres de las columnas basándonos en la fila 'All' (excluyendo el propio 'All')
columnas_ordenadas = tabla_1.loc['All'].drop('All').sort_values(ascending=False).index

#Reorganizo la tabla manteniendo 'All' al final
tabla_1[list(columnas_ordenadas) + ['All']]
tabla_1

De acuerdo a la tabla anterior:  
* El acero A300 parece vulnerable a las deformaciones físicas por impacto (Abolladura) y a las rayaaduras superficiales finas (Rayas).  
* Las rayas severas longitudinales (Surco) están ligadas de forma casi exclusiva al acero A400.
* Las manchas (Mancha) y la suciedad (Suciedad) penalizan más al A400, ya que, concentra 71 de las 72 manchas químicas (Mancha) y 46 de las 55 impurezas (Suciedad).

In [ ]:
#Creo una tabla Espesor vs Tipo de Defecto
from utils.funciones import agrupar_espesores
df['Rango_Espesor'] = df['Espesor'].apply(agrupar_espesores)
tabla_2 = pd.crosstab(df['Rango_Espesor'], df['Tipo_Defecto'], margins=True)

#ORDENO FILAS: Separo la fila 'All', ordeno el resto por la columna 'All' y volvemos a unir
filas_ordenadas = tabla_2.iloc[:-1].sort_values(by='All', ascending=False)
tabla_final = pd.concat([filas_ordenadas, tabla_2.iloc[[-1]]])

#ORDENO COLUMNAS: Busco el orden de las columnas usando la fila 'All' (sin incluir el propio 'All')
columnas_por_cantidad = tabla_final.loc['All'].drop('All').sort_values(ascending=False).index

#Reorganizo las columnas de la tabla final dejando 'All' al extremo derecho
tabla_final[list(columnas_por_cantidad) + ['All']]


De acuerdo a la tabla anterior parece que:   
* En los espesores más finos es más probable tener rayas superficiales.  
* Para espesores medios es más fácil que se produzcan abolladuras.  
* Y en espesores gruesos parece que la visión artificial no es capaz de identificar la mayoría de los defectos, asignándolos a 'Otro'.

## 4.Analysis

### 4.1 Análisis univariante

In [ ]:
# Importamos las nuevas funciones desde tu carpeta utils
from utils.funciones import plot_porcentaje_defectos
plot_porcentaje_defectos(df)

#### CONCLUSIONES  
* El defecto más frecuente con diferencia es 'Otro' con un 34.7%. Más de una de cada tres defectos detectados en planta cae en esta categoría genérica. El sistema automático de visión artificial de la fábrica no es capaz de clasificar con precisión estas imperfecciones. Dejar un volumen tan alto (casi el 35%) sin identificar genera una ceguera operativa enorme. Aquí hay una OPORTUNIDAD millonaria para optimizar y reentrenar los algoritmos de clasificación de la planta.  
* Si limpiamos el dataset y dejamos fuera los errores no clasificados por el software, descubrimos el verdadero foco del problema: solo tres tipos de defectos (Abolladura, Surco y Raya) representan casi el 80% de todas las pérdidas por calidad de la planta (77.5%). Concentrar los esfuerzos de mantenimiento y calibración en estas tres áreas específicas erradicará de forma inmediata el grueso de las mermas en la producción de acero.  
* Las manchas químicas (3.7%) y la suciedad en el acero (2.8%) son los defectos menos frecuentes, sumando apenas un 6.5% entre ambos. Este es un punto muy positivo para la gestión de la planta. Te demuestra que los sistemas de descalillado hidráulico (limpieza con agua a alta presión) y los controles de pureza química en los slabs iniciales están funcionando de forma excelente. La suciedad no es el problema de esta fábrica.

In [ ]:
from utils.funciones import pareto_defectos
#Análisis de Pareto sin 'Other_Faults'
pareto_defectos(df)

Si limpiamos el dataset y dejamos fuera los errores no clasificados por el software, descubrimos el verdadero foco del problema: solo tres tipos de defectos (Abolladura, Surco y Raya) representan casi el 80% de todas las pérdidas por calidad de la planta (77.5%). Concentrar los esfuerzos de mantenimiento y calibración en estas tres áreas específicas erradicará de forma inmediata el grueso de las mermas en la producción de acero.

### 4.2 Análisis bivariante

In [ ]:
from utils.funciones import acero_vs_defecto
# Opción A: Barras separadas (Ideal para ver qué defecto sobresale en cada acero)
acero_vs_defecto(df, tipo_grafico='agrupado')

* En el gráfico anterior vimos que las Abolladuras y los Surcos acumulaban el grueso de los problemas identificados. Al cruzarlos con el tipo de acero, el dato nos da un diagnóstico claro: las Abolladuras son un problema exclusivo del acero blando A300, mientras que los surcos son una patología del acero duro A400.  
* El acero A300 es una aleación físicamente más blanda o maleable. Por lo tanto, cuando pasa por la línea, cualquier variación de presión en los rodillos lo deforma fácilmente provocando abolladuras. Las rayas que sufre son mayoritariamente superficiales y finas (Rayas), causadas por roces ligeros. Prácticamente no tiene rayas severas (Surcos).  
* En el acero A400, el panorama cambia radicalmente. El defecto identificado que más sobresale son las rayas severas (Surco ~34%). Además, es el único acero que presenta un porcentaje visible de manchas.  
* El porcentaje de fallos no clasificados (Otro) se mantiene alto y casi idéntico en ambos aceros (~34% en A300 y ~35% en A400). Esto confirma que el problema de identificación del software de visión artificial es un defecto sistemático de calibración global. No depende del tipo de acero que se esté fabricando; las cámaras fallan por igual a la hora de etiquetar de manera unívoca un tercio de la producción en ambas aleaciones.

In [ ]:
from utils.funciones import espesor_vs_defecto
espesor_vs_defecto(df)

* En las planchas finas, el defecto que predomina en la gráfica es Surco (ocupa más del 40% del total de fallas de este rango). Además, aquí se concentran de manera casi exclusiva las manchas.  
* En el espesor medio, desaparecen por completo las rayas severas (Surco), pero se disparan las abolladuras (con ~27%) y las rayas finas o superficiales (con ~20%).  
* En los bloques gruesos industriales, el gráfico lo ocupa casi todo Otro, que representa más del 70% de todas las mermas de esta categoría. El resto son pequeños porcentajes de Abolladura y Pliegue.  

### 4.3 Análisis multivariante

In [ ]:
from utils.funciones import plot_plano_real_plancha

#Genero la simulación geométrica exacta de la línea de producción
plot_plano_real_plancha(df)


* La línea verde horizontal perfecta que cruza de izquierda a derecha. Está situada exactamente a un ancho de Ancho ≈ 120 mm (cerca del borde superior de la plancha) y se extiende de forma continua a lo largo del proceso. Parece un fallo mecánico sistemático y fijo en la maquinaria.  
* Hay una densa y masiva acumulación de puntos de todos los colores en el extremo izquierdo del mapa, concentrada drásticamente entre los 0 y los 3.000 mm (primeros 3 metros) de largo de las planchas. Pasada esa frontera, la superficie se limpia de forma notable. Delata una severa inestabilidad en el impacto inicial o "cabeza" de la plancha. Cuando el desbaste (slab) de acero incandescente golpea por primera vez los rodillos de deformación, sufre los peores choques térmicos, pérdidas de tracción y desajustes de presión. Una vez que la máquina transforma los primeros 3 metros y el proceso se estabiliza, el acero empieza a salir limpio.  
* El defecto Pliegue (solapamiento de metal) se concentra principalmente en la cabeza (primeros 2 metros) y se dispersa de forma aleatoria por el cuerpo de la plancha, compartiendo exactamente la misma distribución que las Abolladuras. Esto demuestra que es un fallo de origen metalúrgico heredado del desbaste (slab), donde las impurezas o burbujas internas del lingote original se abren y forman dobles capas debido a la enorme presión del proceso de laminación.
* Las rayas finas (Raya) y las manchas químicas (Mancha) se agrupan casi al 100% pegadas a la banda de Ancho = 0. Esto indica problemas de rozamiento con los topes de centrado automático cuando la plancha entra ligeramente desviada por el lateral superior de la mesa de rodillos, acumulando también ahí los líquidos refrigerantes que causan las reacciones químicas en el acero duro.

In [ ]:
from utils.funciones import plot_multivariante_heatmap

#Heatmap (Evaluamos magnitudes físicas e intensidades ópticas juntas)
columnas_interes = ['Espesor', 'Area_Pixel', 'Perimetro_X', 'Indice_Forma_Cuadrada', 'Sum_Luminosidad', 'Indice_Luminosidad']
plot_multivariante_heatmap(df, columnas_interes)